In [148]:
import pandas as pd
import numpy as np

In [149]:
player_full_df = pd.read_csv(
    "../data/processed/player_games.csv"
)

In [150]:
rating_df = player_full_df.copy()

In [151]:
rating_df["twoPointersMade"] = (
    rating_df["fieldGoalsMade"]
    - rating_df["threePointersMade"]
)

rating_df["twoPointersAttempted"] = (
    rating_df["fieldGoalsAttempted"]
    - rating_df["threePointersAttempted"]
)

rating_df["twoPointPercentage"] = (
    rating_df["twoPointersMade"]
    / rating_df["twoPointersAttempted"].replace(0, np.nan)
)

In [152]:
rating_df["minutes"].dtype

<StringDtype(storage='python', na_value=nan)>

In [153]:
rating_df["minutes"].head(10)

0    45:15
1    38:30
2    36:51
3    41:39
4    47:13
5    12:53
6    20:32
7    30:11
8     2:15
9    14:36
Name: minutes, dtype: str

In [154]:
def convert_minutes(value):
    if pd.isna(value):
        return np.nan

    minutes, seconds = value.split(":")
    return int(minutes) + int(seconds) / 60

In [155]:
rating_df["minutesNumeric"] = (
    rating_df["minutes"].apply(convert_minutes)
)

In [156]:
rating_df[
    ["minutes", "minutesNumeric"]
].head(10)

,minutes,minutesNumeric
0,45:15,45.250000
1,38:30,38.500000
2,36:51,36.850000
3,41:39,41.650000
4,47:13,47.216667
5,12:53,12.883333
6,20:32,20.533333
7,30:11,30.183333
8,2:15,2.250000
9,14:36,14.600000


In [157]:
rating_df = rating_df[
    rating_df["minutesNumeric"] > 0
].copy()

In [158]:
weighted_cols = [
    "usagePercentage",
    "assistPercentage",
    "assistToTurnover",
    "assistRatio",
    "turnoverRatio",
    "offensiveReboundPercentage",
    "defensiveReboundPercentage",
    "reboundPercentage",
    "offensiveRating",
    "defensiveRating",
    "netRating",
    "PIE",
]

for col in weighted_cols:
    rating_df[f"{col}Weighted"] = (
        rating_df[col] * rating_df["minutesNumeric"]
    )

In [159]:
player_profile_df = (
    rating_df
    .groupby(
        ["personId", "firstName", "familyName"],
        as_index=False
    )
    .agg(
        gamesPlayed=("gameId", "nunique"),
        totalMinutes=("minutesNumeric", "sum"),

        # scoring volume
        points=("points", "sum"),

        # shooting totals
        fieldGoalsMade=("fieldGoalsMade", "sum"),
        fieldGoalsAttempted=("fieldGoalsAttempted", "sum"),
        threePointersMade=("threePointersMade", "sum"),
        threePointersAttempted=("threePointersAttempted", "sum"),
        freeThrowsMade=("freeThrowsMade", "sum"),
        freeThrowsAttempted=("freeThrowsAttempted", "sum"),
        twoPointersMade=("twoPointersMade", "sum"),
        twoPointersAttempted=("twoPointersAttempted", "sum"),

        assists=("assists", "sum"),
        turnovers=("turnovers", "sum"),
        steals=("steals", "sum"),
        blocks=("blocks", "sum"),
        reboundsOffensive=("reboundsOffensive", "sum"),
        reboundsDefensive=("reboundsDefensive", "sum"),
        reboundsTotal=("reboundsTotal", "sum"),

        usagePercentageWeighted=("usagePercentageWeighted", "sum"),
        assistPercentageWeighted=("assistPercentageWeighted", "sum"),
        assistToTurnoverWeighted=("assistToTurnoverWeighted", "sum"),
        assistRatioWeighted=("assistRatioWeighted", "sum"),
        turnoverRatioWeighted=("turnoverRatioWeighted", "sum"),

        offensiveReboundPercentageWeighted=("offensiveReboundPercentageWeighted", "sum"),
        defensiveReboundPercentageWeighted=("defensiveReboundPercentageWeighted", "sum"),
        reboundPercentageWeighted=("reboundPercentageWeighted", "sum"),

        offensiveRatingWeighted=("offensiveRatingWeighted", "sum"),
        defensiveRatingWeighted=("defensiveRatingWeighted", "sum"),
        netRatingWeighted=("netRatingWeighted", "sum"),
        PIEWeighted=("PIEWeighted", "sum"),
    )
)

In [160]:
player_profile_df["twoPointPercentage"] = (
    player_profile_df["twoPointersMade"]
    / player_profile_df["twoPointersAttempted"].replace(0, np.nan)
)

player_profile_df["threePointersPercentage"] = (
    player_profile_df["threePointersMade"]
    / player_profile_df["threePointersAttempted"].replace(0, np.nan)
)

player_profile_df["fieldGoalPercentage"] = (
    player_profile_df["fieldGoalsMade"]
    / player_profile_df["fieldGoalsAttempted"].replace(0, np.nan)
)

player_profile_df["freeThrowPercentage"] = (
    player_profile_df["freeThrowsMade"]
    / player_profile_df["freeThrowsAttempted"].replace(0, np.nan)
)

player_profile_df["pointsPerMinute"] = (
    player_profile_df["points"]
    / player_profile_df["totalMinutes"]
)

player_profile_df["assistsPerMinute"] = (
    player_profile_df["assists"]
    / player_profile_df["totalMinutes"]
)

player_profile_df["stealsPerMinute"] = (
    player_profile_df["steals"]
    / player_profile_df["totalMinutes"]
)

player_profile_df["blocksPerMinute"] = (
    player_profile_df["blocks"]
    / player_profile_df["totalMinutes"]
)

player_profile_df["threePointAttemptRate"] = (
    player_profile_df["threePointersAttempted"]
    / player_profile_df["fieldGoalsAttempted"].replace(0, np.nan)
)

player_profile_df["freeThrowRate"] = (
    player_profile_df["freeThrowsAttempted"]
    / player_profile_df["fieldGoalsAttempted"].replace(0, np.nan)
)

player_profile_df["trueShootingPercentage"] = (
    player_profile_df["points"]
    / (
        2 * (
            player_profile_df["fieldGoalsAttempted"]
            + 0.44 * player_profile_df["freeThrowsAttempted"]
        )
    ).replace(0, np.nan)
)

for col in weighted_cols:
    player_profile_df[col] = (
        player_profile_df[f"{col}Weighted"]
        / player_profile_df["totalMinutes"]
    )

In [161]:
player_profile_df.head()

,personId,firstName,familyName,gamesPlayed,totalMinutes,points,fieldGoalsMade,fieldGoalsAttempted,threePointersMade,threePointersAttempted,...,assistToTurnover,assistRatio,turnoverRatio,offensiveReboundPercentage,defensiveReboundPercentage,reboundPercentage,offensiveRating,defensiveRating,netRating,PIE
0,201142,Kevin,Durant,1,47.050000,23,9,16,0,4,...,0.75,12.0,16.0,0.000,0.167,0.090,107.6,110.0,-2.4,0.121
1,201143,Al,Horford,1,20.250000,5,2,7,1,4,...,0.50,10.0,20.0,0.000,0.385,0.161,104.7,122.2,-17.6,0.000
2,201144,Mike,Conley,1,12.716667,3,1,5,1,2,...,0.00,0.0,16.7,0.067,0.100,0.080,100.0,123.1,-23.1,-0.037
3,201566,Russell,Westbrook,1,18.800000,6,2,8,0,2,...,0.50,8.3,16.7,0.056,0.227,0.150,121.1,110.0,11.1,0.052
4,201939,Stephen,Curry,1,32.066667,23,6,14,3,9,...,2.00,17.4,8.7,0.000,0.038,0.018,112.9,108.3,4.5,0.135


In [162]:
player_profile_df[
    [
        "firstName",
        "familyName",
        "gamesPlayed",
        "totalMinutes",
        "twoPointPercentage",
        "threePointersPercentage",
        "pointsPerMinute",
        "assistsPerMinute",
        "reboundPercentage",
        "defensiveRating",
    ]
].head(20)

,firstName,familyName,gamesPlayed,totalMinutes,twoPointPercentage,threePointersPercentage,pointsPerMinute,assistsPerMinute,reboundPercentage,defensiveRating
0,Kevin,Durant,1,47.050000,0.750000,0.000000,0.488842,0.063762,0.090,110.0
1,Al,Horford,1,20.250000,0.333333,0.250000,0.246914,0.049383,0.161,122.2
2,Mike,Conley,1,12.716667,0.000000,0.500000,0.235911,0.000000,0.080,123.1
3,Russell,Westbrook,1,18.800000,0.333333,0.000000,0.319149,0.053191,0.150,110.0
4,Stephen,Curry,1,32.066667,0.600000,0.333333,0.717256,0.124740,0.018,108.3
5,DeMar,DeRozan,1,37.416667,0.733333,0.500000,0.775056,0.240535,0.075,107.2
6,Jrue,Holiday,1,33.066667,0.500000,0.142857,0.423387,0.211694,0.085,102.7
7,Bismack,Biyombo,1,4.933333,NaN,NaN,0.000000,0.000000,0.143,90.9
8,Klay,Thompson,1,22.333333,0.375000,0.200000,0.447761,0.044776,0.043,133.3
9,Jimmy,Butler III,1,34.733333,0.500000,0.500000,0.892514,0.115163,0.082,104.1


In [163]:
def percentile_score(series):
    return series.rank(pct=True) * 100

def weighted_rating(components):
    numerator = 0
    denominator = 0

    for series, weight in components:
        valid = series.notna()

        numerator += series.fillna(0) * weight
        denominator += valid.astype(float) * weight

    return numerator / denominator.replace(0, np.nan)

In [164]:
player_profile_df["finishing"] = weighted_rating([
    (
        percentile_score(
            player_profile_df["twoPointPercentage"]
        ),
        0.40
    ),
    (
        percentile_score(
            player_profile_df["freeThrowRate"]
        ),
        0.25
    ),
    (
        percentile_score(
            player_profile_df["pointsPerMinute"]
        ),
        0.20
    ),
    (
        percentile_score(
            player_profile_df["trueShootingPercentage"]
        ),
        0.15
    ),
])

player_profile_df["shooting"] = weighted_rating([
    (
        percentile_score(
            player_profile_df["threePointersPercentage"]
        ),
        0.40
    ),
    (
        percentile_score(
            player_profile_df["threePointAttemptRate"]
        ),
        0.25
    ),
    (
        percentile_score(
            player_profile_df["trueShootingPercentage"]
        ),
        0.20
    ),
    (
        percentile_score(
            player_profile_df["freeThrowPercentage"]
        ),
        0.15
    ),
])

player_profile_df["playmaking"] = weighted_rating([
    (
        percentile_score(
            player_profile_df["assistPercentage"]
        ),
        0.40
    ),
    (
        percentile_score(
            player_profile_df["assistToTurnover"]
        ),
        0.25
    ),
    (
        percentile_score(
            player_profile_df["assistRatio"]
        ),
        0.20
    ),
    (
        100 - percentile_score(
            player_profile_df["turnoverRatio"]
        ),
        0.15
    ),
])

player_profile_df["rebounding"] = weighted_rating([
    (
        percentile_score(
            player_profile_df["reboundPercentage"]
        ),
        0.45
    ),
    (
        percentile_score(
            player_profile_df["offensiveReboundPercentage"]
        ),
        0.30
    ),
    (
        percentile_score(
            player_profile_df["defensiveReboundPercentage"]
        ),
        0.25
    ),
])

player_profile_df["defense"] = weighted_rating([
    (
        percentile_score(
            player_profile_df["stealsPerMinute"]
        ),
        0.30
    ),
    (
        percentile_score(
            player_profile_df["blocksPerMinute"]
        ),
        0.30
    ),
    (
        100 - percentile_score(
            player_profile_df["defensiveRating"]
        ),
        0.40
    ),
])

In [165]:
ratings_df = player_profile_df[
    [
        "personId",
        "firstName",
        "familyName",
        "gamesPlayed",
        "totalMinutes",
        "finishing",
        "shooting",
        "playmaking",
        "defense",
        "rebounding",
    ]
].copy()

ratings_df[
    [
        "finishing",
        "shooting",
        "playmaking",
        "defense",
        "rebounding",
    ]
].isna().sum()

finishing      0
shooting      14
playmaking     0
defense        0
rebounding     0
dtype: int64

In [166]:
rating_cols = [
    "finishing",
    "shooting",
    "playmaking",
    "defense",
    "rebounding",
]

ratings_df[rating_cols] = (
    ratings_df[rating_cols]
    .round(1)
)

In [167]:
ratings_df.sort_values(
    "shooting",
    ascending=False
).head(20)

,personId,firstName,familyName,gamesPlayed,totalMinutes,finishing,shooting,playmaking,defense,rebounding
147,1631127,Harrison,Ingram,1,4.066667,61.5,98.2,25.5,55.4,72.4
144,1631108,Max,Christie,1,25.950000,46.4,96.1,42.4,44.5,11.2
201,1642856,Egor,Dëmin,1,22.316667,74.6,91.8,57.9,46.8,60.5
200,1642851,Kon,Knueppel,1,25.483333,47.4,91.5,34.6,50.3,51.8
210,1642954,Will,Richard,1,13.583333,65.4,89.1,59.3,36.6,42.9
68,1629013,Landry,Shamet,1,13.816667,53.2,88.3,55.9,63.9,39.2
91,1629731,Dean,Wade,1,28.116667,74.0,85.9,25.5,67.7,22.9
80,1629611,Terance,Mann,1,19.450000,74.5,83.3,45.5,37.3,30.0
112,1630540,Miles,McBride,1,25.866667,49.5,83.2,36.1,63.0,32.5
76,1629060,Rui,Hachimura,1,35.416667,38.1,82.9,65.9,28.7,38.2


In [168]:
player_games_df = pd.read_csv(
    "../data/processed/player_games_two_seasons.csv",
    dtype={"gameId": str},
)

player_games_df["gameDate"] = pd.to_datetime(
    player_games_df["gameDate"]
)

In [169]:
rating_date = player_games_df["gameDate"].max()

rating_date

Timestamp('2026-04-12 00:00:00')

In [170]:
player_games_df["daysAgo"] = (
    rating_date - player_games_df["gameDate"]
).dt.days

In [171]:
HALF_LIFE_DAYS = 180

player_games_df["recencyWeight"] = (
    0.5 ** (
        player_games_df["daysAgo"]
        / HALF_LIFE_DAYS
    )
)

In [172]:
player_games_df[
    [
        "gameDate",
        "daysAgo",
        "recencyWeight",
    ]
].sort_values(
    "gameDate",
    ascending=False,
).head()

,gameDate,daysAgo,recencyWeight
64693,2026-04-12,0,1.0
64405,2026-04-12,0,1.0
64431,2026-04-12,0,1.0
64432,2026-04-12,0,1.0
64433,2026-04-12,0,1.0


In [173]:
player_games_df[
    [
        "gameDate",
        "daysAgo",
        "recencyWeight",
    ]
].sort_values(
    "gameDate"
).head()

,gameDate,daysAgo,recencyWeight
0,2024-10-22,537,0.126452
29,2024-10-22,537,0.126452
30,2024-10-22,537,0.126452
31,2024-10-22,537,0.126452
32,2024-10-22,537,0.126452


In [174]:
import pandas as pd

ratings_df = pd.read_csv(
    "../data/processed/player_ratings.csv"
)

In [175]:
ratings_df.shape

(687, 62)

In [176]:
ratings_df[
    [
        "finishing",
        "shooting",
        "playmaking",
        "defense",
        "rebounding",
    ]
].describe()

,finishing,shooting,playmaking,defense,rebounding
count,687.000000,684.000000,687.000000,687.000000,687.00000
mean,49.860408,49.633772,50.052402,50.013100,50.07409
std,21.212461,20.159435,21.816482,17.690203,26.96884
min,0.700000,1.500000,1.300000,1.900000,0.90000
25%,34.350000,33.775000,31.900000,38.050000,26.60000
50%,51.500000,49.950000,49.500000,50.500000,49.50000
75%,66.650000,64.500000,68.600000,62.750000,72.25000
max,98.800000,97.100000,97.200000,94.900000,99.00000


In [177]:
ratings_df[
    ratings_df["shooting"].isna()
][
    [
        "firstName",
        "familyName",
        "gamesPlayed",
        "totalMinutes",
        "finishing",
        "shooting",
        "playmaking",
        "defense",
        "rebounding",
    ]
]

,firstName,familyName,gamesPlayed,totalMinutes,finishing,shooting,playmaking,defense,rebounding
80,Jahlil,Okafor,1,3.366667,0.7,NaN,70.7,41.9,59.5
231,Jalen,McDaniels,4,7.350000,0.7,NaN,67.6,70.8,0.9
595,Jesse,Edwards,2,4.666667,0.7,NaN,68.3,2.0,0.9


In [178]:
ratings_df.sort_values(
    "finishing",
    ascending=False,
)[
    [
        "firstName",
        "familyName",
        "gamesPlayed",
        "totalMinutes",
        "finishing",
        "shooting",
        "playmaking",
        "defense",
        "rebounding",
    ]
].head(20)

,firstName,familyName,gamesPlayed,totalMinutes,finishing,shooting,playmaking,defense,rebounding
371,Alex,Antetokounmpo,6,20.633333,98.8,57.8,17.3,67.4,73.3
384,Jalen,Duren,148,4009.433333,93.1,41.3,47.1,59.4,97.5
53,Giannis,Antetokounmpo,103,3327.850000,92.1,29.9,81.6,52.1,91.8
135,Jarrett,Allen,138,3815.266667,91.8,28.1,39.4,61.3,92.6
521,Norchad,Omier,6,23.666667,91.8,38.6,42.8,24.2,89.7
226,Daniel,Gafford,112,2420.600000,91.6,39.0,27.6,55.2,90.0
76,Nikola,Jokić,135,4835.666667,90.5,69.2,90.2,53.9,93.7
634,Joan,Beringer,40,314.133333,88.3,39.6,22.8,42.8,86.0
208,Zion,Williamson,92,2698.350000,88.0,27.8,70.5,48.6,68.1
164,Shai,Gilgeous-Alexander,144,4856.600000,87.6,67.7,88.0,83.1,31.8


In [179]:
ratings_df.sort_values(
    "shooting",
    ascending=False,
)[
    [
        "firstName",
        "familyName",
        "gamesPlayed",
        "totalMinutes",
        "shooting",
    ]
].head(20)

,firstName,familyName,gamesPlayed,totalMinutes,shooting
3,P.J.,Tucker,3,57.866667,97.1
650,Koby,Brea,12,83.866667,93.5
265,Isaiah,Joe,145,3111.433333,92.8
427,Caleb,Houstan,76,863.466667,92.8
612,Cormac,Ryan,11,270.683333,92.0
442,AJ,Green,151,3929.066667,91.4
131,Luke,Kennard,143,3152.250000,91.3
568,Cam,Spencer,97,1966.666667,91.0
330,Sam,Hauser,149,3475.366667,90.5
492,DaRon,Holmes II,25,209.516667,89.8


In [180]:
ratings_df.sort_values(
    "playmaking",
    ascending=False,
)[
    [
        "firstName",
        "familyName",
        "gamesPlayed",
        "totalMinutes",
        "playmaking",
    ]
].head(20)

,firstName,familyName,gamesPlayed,totalMinutes,playmaking
249,Tyrese,Haliburton,73,2450.433333,97.2
266,Tre,Jones,111,2657.366667,93.1
568,Cam,Spencer,97,1966.666667,92.5
263,Immanuel,Quickley,103,3149.100000,91.9
79,T.J.,McConnell,135,2377.966667,91.2
81,Tyus,Jones,148,3152.133333,90.7
1,Chris,Paul,98,2519.700000,90.5
76,Nikola,Jokić,135,4835.666667,90.2
159,Jalen,Brunson,139,4890.633333,89.6
35,Jimmy,Butler III,93,2927.833333,88.9


In [181]:
ratings_df.sort_values(
    "defense",
    ascending=False,
)[
    [
        "firstName",
        "familyName",
        "gamesPlayed",
        "totalMinutes",
        "defense",
    ]
].head(20)

,firstName,familyName,gamesPlayed,totalMinutes,defense
55,Thanasis,Antetokounmpo,34,147.950000,94.9
264,Paul,Reed,110,1338.183333,94.6
236,Matisse,Thybulle,45,791.966667,91.8
372,Olivier,Sarr,4,39.350000,91.1
511,Johni,Broome,11,54.750000,90.9
678,Chris,Mañon,9,46.083333,90.4
358,Stanley,Umude,24,92.350000,90.2
686,Jayson,Kent,5,22.466667,90.0
128,Jonathan,Isaac,123,1610.966667,89.9
297,Josh,Christopher,14,69.350000,88.2


In [182]:
ratings_df.sort_values(
    "rebounding",
    ascending=False,
)[
    [
        "firstName",
        "familyName",
        "gamesPlayed",
        "totalMinutes",
        "rebounding",
    ]
].head(20)

,firstName,familyName,gamesPlayed,totalMinutes,rebounding
39,Andre,Drummond,103,1981.433333,99.0
586,N'Faly,Dante,8,66.033333,98.4
404,Oscar,Tshiebwe,41,703.600000,98.4
555,Donovan,Clingan,144,3417.383333,98.1
176,Mitchell,Robinson,77,1464.683333,98.1
25,Jonas,Valančiūnas,146,2394.850000,98.0
384,Jalen,Duren,148,4009.433333,97.5
395,Walker,Kessler,63,1894.300000,97.4
13,DeAndre,Jordan,68,889.483333,97.1
100,Domantas,Sabonis,89,2992.500000,97.1


In [183]:
qualified_df = ratings_df[
    (ratings_df["gamesPlayed"] >= 20)
    & (ratings_df["totalMinutes"] >= 300)
].copy()

In [184]:
qualified_df.sort_values(
    "shooting",
    ascending=False
).head(20)

,personId,firstName,familyName,gamesPlayed,totalMinutes,points,fieldGoalsMade,fieldGoalsAttempted,threePointersMade,threePointersAttempted,...,reboundPercentage,offensiveRating,defensiveRating,netRating,PIE,finishing,shooting,playmaking,rebounding,defense
427,1631216,Caleb,Houstan,76,863.466667,281,94,216,75,181,...,0.048147,103.724647,107.692354,-3.952990,0.057009,42.5,92.8,29.5,15.9,50.7
265,1630198,Isaiah,Joe,145,3111.433333,1541,506,1131,373,894,...,0.057770,118.380678,104.784633,13.597832,0.094220,62.4,92.8,41.8,24.8,55.6
442,1631260,AJ,Green,151,3929.066667,1350,444,1042,387,917,...,0.047985,113.650299,112.987371,0.661202,0.057771,27.8,91.4,45.3,11.2,24.7
131,1628379,Luke,Kennard,143,3152.250000,1228,442,874,230,506,...,0.056148,115.926079,114.018302,1.911316,0.092441,58.1,91.3,71.0,23.3,33.9
568,1642285,Cam,Spencer,97,1966.666667,904,297,638,165,377,...,0.051172,113.132448,115.224934,-2.096194,0.116801,54.9,91.0,92.5,18.5,28.6
330,1630573,Sam,Hauser,149,3475.366667,1324,466,1075,364,903,...,0.072127,119.973846,110.808698,9.172439,0.080731,52.7,90.5,28.7,41.9,41.3
281,1630241,Sam,Merrill,123,2778.416667,1172,395,908,295,743,...,0.050053,119.028054,109.988970,9.037728,0.073027,60.0,89.6,53.2,16.6,47.8
628,1642857,Kasparas,Jakučionis,53,944.650000,328,102,238,66,156,...,0.067566,115.977581,113.306023,2.667872,0.072147,40.1,89.5,72.0,41.2,40.3
12,201587,Nicolas,Batum,152,2662.050000,606,200,476,180,431,...,0.073693,112.237544,109.318233,2.925659,0.064205,20.8,88.9,32.2,42.4,70.0
110,1627752,Taurean,Prince,106,2777.183333,895,321,705,208,475,...,0.066165,114.084126,115.299740,-1.203365,0.064413,30.9,88.8,44.9,31.3,32.3


In [185]:
profiles_df = pd.read_csv(
    "../data/processed/player_profiles.csv"
)

print(profiles_df.shape)
print(profiles_df.isna().sum())

profiles_df["position"].value_counts(dropna=False)

(687, 7)
personId        0
firstName       0
familyName      0
position        0
heightInches    0
weight          6
birthDate       0
dtype: int64


position
Guard             291
Forward           214
Center             70
Guard-Forward      45
Forward-Center     33
Center-Forward     21
Forward-Guard      13
Name: count, dtype: int64

In [186]:
def map_physical_group(position):
    if position == "Guard":
        return "Guard"

    if position in [
        "Guard-Forward",
        "Forward-Guard",
    ]:
        return "Wing"

    if position == "Forward":
        return "Forward"

    if position in [
        "Forward-Center",
        "Center-Forward",
        "Center",
    ]:
        return "Big"

    return "Unknown"

In [187]:
profiles_df["physicalGroup"] = (
    profiles_df["position"]
    .apply(map_physical_group)
)

In [188]:
profiles_df["physicalGroup"].value_counts()

physicalGroup
Guard      291
Forward    214
Big        124
Wing        58
Name: count, dtype: int64

In [189]:
profiles_df[
    profiles_df["weight"].isna()
][
    [
        "firstName",
        "familyName",
        "position",
        "heightInches",
        "weight",
    ]
]

,firstName,familyName,position,heightInches,weight
587,Jaylen,Wells,Forward,79,NaN
605,Tolu,Smith,Forward,83,NaN
671,Chris,Youngblood,Guard,76,NaN
677,LJ,Cryer,Guard,72,NaN
682,Lawson,Lovering,Center,84,NaN
683,Jahmyl,Telfort,Guard,79,NaN


In [190]:
profiles_df["weight"] = (
    profiles_df["weight"]
    .fillna(
        profiles_df.groupby(
            "physicalGroup"
        )["weight"].transform("median")
    )
)

In [191]:
profiles_df["heightPercentile"] = (
    profiles_df
    .groupby("physicalGroup")["heightInches"]
    .rank(pct=True)
    * 100
)

profiles_df["weightPercentile"] = (
    profiles_df
    .groupby("physicalGroup")["weight"]
    .rank(pct=True)
    * 100
)

profiles_df["physical"] = (
    0.55 * profiles_df["heightPercentile"]
    + 0.45 * profiles_df["weightPercentile"]
)

In [192]:
profiles_df[
    [
        "firstName",
        "familyName",
        "position",
        "physicalGroup",
        "heightInches",
        "weight",
        "physical",
    ]
].sort_values(
    "physical",
    ascending=False
).head(30)

,firstName,familyName,position,physicalGroup,heightInches,weight,physical
490,Zach,Edey,Center,Big,87,305.0,99.112903
99,Ben,Simmons,Guard-Forward,Wing,82,240.0,98.836207
314,Johnny,Juzang,Guard,Guard,79,226.0,97.113402
639,Danny,Wolf,Forward,Forward,83,250.0,96.939252
555,Donovan,Clingan,Center,Big,86,280.0,96.108871
641,Adou,Thiero,Guard,Guard,79,220.0,95.644330
613,Alex,Ducas,Guard,Guard,79,220.0,95.644330
328,Trendon,Watford,Guard-Forward,Wing,80,245.0,95.258621
474,Jett,Howard,Guard,Guard,80,215.0,94.716495
374,Paolo,Banchero,Forward,Forward,82,250.0,94.240654


In [193]:
player_games_df = pd.read_csv(
    "../data/processed/player_games_two_seasons.csv",
    dtype={"gameId": str},
)

player_games_df["gameDate"] = pd.to_datetime(
    player_games_df["gameDate"]
)

In [194]:
def convert_minutes(value):
    if pd.isna(value):
        return np.nan

    minutes, seconds = value.split(":")
    return int(minutes) + int(seconds) / 60

In [195]:
def add_expected_minutes(player_games_df):
    df = player_games_df.copy()

    df["minutesNumeric"] = (
        df["minutes"]
        .apply(convert_minutes)
    )

    df = df.sort_values(
        ["personId", "gameDate"]
    )

    df["expectedMinutes"] = (
        df
        .groupby("personId")["minutesNumeric"]
        .transform(
            lambda s: (
                s.shift(1)
                .rolling(
                    window=10,
                    min_periods=1,
                )
                .mean()
            )
        )
    )

    return df

In [196]:
test_df = add_expected_minutes(
    player_games_df
)

In [202]:
ratings_df = pd.read_csv(
    "../data/processed/player_ratings.csv"
)

In [203]:
RATING_COLS = [
    "finishing",
    "shooting",
    "playmaking",
    "defense",
    "rebounding",
    "physical",
]

test_df = test_df.merge(
    ratings_df[
        [
            "personId",
            *RATING_COLS,
        ]
    ],
    on="personId",
    how="left",
    validate="many_to_one",
)

In [205]:
test_df[
    [
        "firstName",
        "familyName",
        "expectedMinutes",
        "finishing",
        "shooting",
        "physical",
    ]
].head()

,firstName,familyName,expectedMinutes,finishing,shooting,physical
0,LeBron,James,NaN,77.1,50.6,87.4
1,LeBron,James,34.650000,77.1,50.6,87.4
2,LeBron,James,34.675000,77.1,50.6,87.4
3,LeBron,James,34.372222,77.1,50.6,87.4
4,LeBron,James,34.729167,77.1,50.6,87.4


In [207]:
test_df["isStarter"] = (
    test_df["position"].notna()
)

test_df["expectedMinutes"] = (
    test_df["expectedMinutes"]
    .fillna(
        test_df["isStarter"].map({
            True: 30.0,
            False: 12.0,
        })
    )
)

In [208]:
missing_ratings = test_df[
    test_df[RATING_COLS]
    .isna()
    .all(axis=1)
][
    [
        "personId",
        "firstName",
        "familyName",
        "teamTricode",
        "gameId",
        "minutes",
        "comment",
        "isStarter",
        "expectedMinutes",
    ]
]

missing_ratings

,personId,firstName,familyName,teamTricode,gameId,minutes,comment,isStarter,expectedMinutes
10717,1626174,Christian,Wood,LAL,0022400118,NaN,DND - Injury/Illness,False,12.0
10718,1626174,Christian,Wood,LAL,0022400137,NaN,DND - Injury/Illness,False,12.0
10719,1626174,Christian,Wood,LAL,0022400742,NaN,DND - Injury/Illness,False,12.0
32147,1630284,Kevon,Harris,ATL,0022400878,NaN,DNP - Coach's Decision,False,12.0
32148,1630284,Kevon,Harris,HOU,0022500001,NaN,DNP - Coach's Decision,False,12.0
32149,1630284,Kevon,Harris,HOU,0022500160,NaN,DNP - Coach's Decision,False,12.0
34981,1630554,Jason,Preston,UTA,0022400084,NaN,DNP - Coach's Decision,False,12.0
38790,1630647,Eugene,Omoruyi,TOR,0022400550,NaN,DNP - Coach's Decision,False,12.0
54616,1641907,Erik,Stevenson,WAS,0022400799,NaN,DNP - Coach's Decision,False,12.0
54775,1642013,Malik,Williams,ATL,0022500411,NaN,DNP - Coach's Decision,False,12.0


In [197]:
test_df[
    test_df["familyName"] == "Jokić"
][
    [
        "gameDate",
        "minutes",
        "minutesNumeric",
        "expectedMinutes",
    ]
].head(15)

,gameDate,minutes,minutesNumeric,expectedMinutes
421,2024-10-24,35:12,35.200000,NaN
1011,2024-10-26,36:46,36.766667,35.200000
1288,2024-10-28,43:41,43.683333,35.983333
1450,2024-10-29,40:31,40.516667,38.550000
2090,2024-11-01,39:42,39.700000,39.041667
2270,2024-11-02,29:42,29.700000,39.173333
2626,2024-11-04,38:12,38.200000,37.594444
2972,2024-11-06,39:29,39.483333,37.680952
3514,2024-11-08,39:60,40.000000,37.906250
3974,2024-11-10,38:00,38.000000,38.138889
